# 🌍📰 GeoBrief · CrewAI + Ollama · Noticias ONU (RSS)
**Objetivo:** crear un boletín de geopolítica con tres agentes coordinados y un LLM local.

**Cambio de fuente:** esta versión obtiene noticias del feed RSS de Noticias ONU en español. RSS es una fuente XML por HTTP; ya no consulta la API JSON de GDELT. El cambio permite continuar el taller ante sus errores 429. La orquestación con CrewAI y Ollama se mantiene.

Conservamos los pasos de SportsCrew: preparación → tema → descarga → modelo → agentes → informe.

**Aprenderás:** `LLM`, `Agent`, `Task`, herramientas, `context` y `Process.sequential`.

**Resultado:** dossier, análisis y boletín de hasta 200 palabras con referencias. La KB contiene los resúmenes publicados en el feed, no artículos completos. Todas las noticias provienen de una única fuente institucional: no hay contraste independiente entre medios.

**Duración orientativa:** 90–120 minutos, con modelo y dependencias preparados antes. Python básico. Ejecuta en Jupyter local o VS Code. En Colab, `localhost` no apunta a tu ordenador.

**Peticiones:** una descarga del feed en la primera preparación; las siguientes ejecuciones usan la copia local. No se descargan páginas de artículos ni se hacen reintentos automáticos. Ollama procesa el contexto localmente. Los enlaces se incluyen como referencias para revisión manual.


## 1) Instalar dependencias (una vez)
En una terminal:
```bash
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install jupyterlab
python -m jupyter lab
```
La celda siguiente instala CrewAI con LiteLLM y requests.

# Instala dependencias
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "crewai[litellm]", "requests"])
print("Dependencias instaladas. Reinicia el kernel.")

## 2) Preparar Ollama
Instala [Ollama](https://ollama.com/download) y descarga el modelo:
```bash
ollama pull llama3.1:latest
```
Abre la aplicación Ollama o ejecuta `ollama serve` en otra terminal.

In [1]:
# Configuración inicial: carga librerías y desactiva telemetría
import os
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_TELEMETRY_DISABLED"] = "true"

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
import requests
import xml.etree.ElementTree as ET
from IPython.display import Markdown, display
from crewai import Agent, Task, Crew, LLM, Process
from crewai.tools import tool

## 3) Configurar el taller
- `KEYWORDS`: filtra noticias por tema
- `MAX_NEWS=3`: máximo de noticias
- `OFFLINE=True`: usa copia local sin descargar
- `USE_TOOL=True`: el investigador usa la herramienta

In [2]:
# Configuración del taller: define tema, palabras clave, límites y rutas de archivos
TOPIC = "Diplomacia, conflictos y cooperación internacional"
FEED_URL = "https://news.un.org/feed/subscribe/es/news/all/rss.xml"
KEYWORDS = ["diplomacia", "conflicto", "paz", "guerra", "asamblea", "seguridad",
            "gaza", "israel", "ucrania", "rusia", "iran", "sudan", "refugiados"]
MAX_NEWS = 3
OFFLINE = False
USE_TOOL = True
MODEL_NAME = "llama3.1:latest"
OLLAMA_URL = "http://localhost:11434"

# TOPIC = "Gaza y la respuesta internacional"
# KEYWORDS = ["gaza", "israel", "palestina"]

DATA_DIR = Path("geobrief_datos")
DATA_DIR.mkdir(exist_ok=True)
feed_id = hashlib.sha256(FEED_URL.encode()).hexdigest()[:10]
FEED_FILE = DATA_DIR / f"rss_{feed_id}.json"
ATTEMPT_FILE = DATA_DIR / f"rss_intento_{feed_id}.json"
CACHE_FILE = DATA_DIR / "kb_rss_actual.json"
settings = dict(source="Noticias ONU RSS", url=FEED_URL, keywords=KEYWORDS, max_news=MAX_NEWS)
print("Tema:", TOPIC)
print("Copia del feed:", FEED_FILE.resolve())
print("KB de esta selección:", CACHE_FILE.resolve())

Tema: Diplomacia, conflictos y cooperación internacional
Copia del feed: /Users/carlos/work/ponencias/talleres/Unex26/geobrief_datos/rss_b17c36dec5.json
KB de esta selección: /Users/carlos/work/ponencias/talleres/Unex26/geobrief_datos/kb_rss_actual.json


## 4) Leer el feed y construir la KB
Descarga el feed RSS de Noticias ONU, extrae resúmenes y filtra por palabras clave. Guarda copia local para reutilizar.

In [3]:
# Funciones simplificadas para procesar RSS
def save_json(path, data):
    """Guarda datos en JSON."""
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def clean_text(value):
    """Elimina etiquetas HTML de un texto."""
    import re
    return re.sub(r'<[^>]+>', '', value or "").strip()

def parse_rss(xml_bytes):
    """Lee un RSS y extrae título, URL, fecha y resumen."""
    root = ET.fromstring(xml_bytes)
    entries = []
    for item in root.findall("./channel/item"):
        title = clean_text(item.findtext("title"))
        url = (item.findtext("link") or "").strip()
        summary = clean_text(item.findtext("description"))
        if not title or len(summary) < 60 or not url.startswith("http"):
            continue
        published = item.findtext("pubDate") or "No disponible"
        entries.append({
            "title": title,
            "url": url,
            "source": "Noticias ONU",
            "published_at": published,
            "text": summary,
            "text_type": "Resumen del feed RSS"
        })
    if not entries:
        raise RuntimeError("El feed no contiene entradas utilizables.")
    return entries

def load_feed():
    """Lee la caché o descarga el feed una vez."""
    if FEED_FILE.exists():
        print("Feed local: cero peticiones.")
        return json.loads(FEED_FILE.read_text(encoding="utf-8"))
    
    if OFFLINE:
        raise RuntimeError("Modo offline: falta FEED_FILE.")
    
    if ATTEMPT_FILE.exists():
        raise RuntimeError("Ya se intentó la descarga. Revisa " + str(ATTEMPT_FILE))
    
    downloaded_at = datetime.now(timezone.utc).isoformat()
    response = requests.get(FEED_URL, timeout=30)
    
    if response.status_code != 200:
        save_json(ATTEMPT_FILE, {"status": response.status_code})
        raise RuntimeError(f"RSS: HTTP {response.status_code}")
    
    data = {"downloaded_at_utc": downloaded_at, "feed_url": FEED_URL, "entries": parse_rss(response.content)}
    save_json(FEED_FILE, data)
    print("Feed descargado y guardado.")
    return data

In [4]:
# Filtra noticias por palabras clave y construye la KB
feed_snapshot = load_feed()
KB = []

for entry in feed_snapshot["entries"]:
    texto = (entry["title"] + " " + entry["text"]).lower()
    if KEYWORDS and not any(kw.lower() in texto for kw in KEYWORDS):
        continue
    KB.append({**entry, "id": f"N{len(KB) + 1}"})
    if len(KB) >= MAX_NEWS:
        break

if not KB:
    raise RuntimeError("Sin coincidencias. Cambia KEYWORDS o usa [].")

snapshot = {"downloaded_at_utc": feed_snapshot["downloaded_at_utc"], "settings": settings, "documents": KB}
save_json(CACHE_FILE, snapshot)
print("Documentos:", len(KB))

Feed local: cero peticiones.
Documentos: 3


### Revisar la KB
Verifica las noticias antes de continuar.

In [5]:
# Muestra las noticias de la KB
for doc in KB:
    print(f"[{doc['id']}] {doc['title']}")
    print(f"URL: {doc['url']}\n")

[N1] El Niño pone a prueba la capacidad del mundo para adaptarse al cambio climático
URL: https://news.un.org/feed/view/es/story/2026/09/1541945

[N2] Guterres alerta de un mundo en plena reconfiguración del poder: entre Estados, corporaciones y máquinas
URL: https://news.un.org/feed/view/es/story/2026/09/1541943

[N3] Minuto a minuto de la Asamblea General UNGA81
URL: https://news.un.org/feed/view/es/story/2026/09/1541929



## 5) Crear herramienta para consultar KB
Una función que los agentes pueden invocar para leer las noticias.

In [6]:
# Crea la herramienta para consultar la KB
def format_kb(documents):
    """Formatea la KB como texto."""
    bloques = []
    for doc in documents:
        bloques.append(f"[{doc['id']}] {doc['title']}\nURL: {doc['url']}\n{doc['text']}")
    return "\n\n---\n\n".join(bloques)

KB_TEXT = format_kb(KB)

@tool("consultar_kb")
def consultar_kb() -> str:
    """Devuelve las noticias de la KB. Sin argumentos."""
    return KB_TEXT

print("Contexto:", len(KB_TEXT), "caracteres")

Contexto: 1845 caracteres


## 6) Configurar modelo Ollama
Verifica que Ollama está activo y configura el modelo local con CrewAI.

In [7]:
# Configura y prueba el modelo Ollama
response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10)
if response.status_code != 200:
    raise RuntimeError("No se puede conectar con Ollama. Abre la app o ejecuta ollama serve.")

models = [m["name"] for m in response.json().get("models", [])]
if MODEL_NAME not in models:
    raise RuntimeError(f"Falta el modelo. Ejecuta: ollama pull {MODEL_NAME}")

llm = LLM(
    model=f"openai/{MODEL_NAME}",
    base_url=f"{OLLAMA_URL}/v1",
    api_key="ollama",
    temperature=0.2,
    max_tokens=1800,
)
print(llm.call("Responde: Modelo listo."))

¡Listo! ¿En qué puedo ayudarte?


## 7) Crear los tres agentes
Investigador: extrae hechos con citas. Analista: relaciona hallazgos. Editor: redacta boletín claro.

In [8]:
# Crea los tres agentes
RULES = """
Escribe en español. Usa solo las noticias proporcionadas.
No inventes datos. Cita los identificadores [N1], [N2], etc.
Distingue entre hechos reportados e interpretaciones.
Solo has leído resúmenes RSS, no artículos completos.
"""

researcher = Agent(
    role="Investigador",
    goal="Preparar un dossier breve con hechos relevantes y citas [N#].",
    backstory="Seleccionas hechos reportados y actores. " + RULES,
    llm=llm,
    tools=[consultar_kb] if USE_TOOL else [],
    allow_delegation=False,
    verbose=True,
)

analyst = Agent(
    role="Analista",
    goal="Relacionar hallazgos y señalar límites.",
    backstory="Comparas fuentes y dudas. " + RULES,
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

editor = Agent(
    role="Editor",
    goal="Redactar un boletín claro respaldado por las fuentes.",
    backstory="Eliminas afirmaciones sin respaldo. " + RULES,
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

## 8) Definir tareas y Crew
Las tareas se ejecutan en orden: investigación → análisis → edición. Cada una recibe el contexto de la anterior.

In [9]:
# Define las tareas y crea la Crew
research_input = "Llama a consultar_kb para leer las noticias." if USE_TOOL else f"KB: {KB_TEXT}"

research_task = Task(
    description=f"Tema: {TOPIC}. {research_input}\n{RULES}\nPrepara un dossier de hasta 180 palabras con hechos y citas [N#].",
    expected_output="Dossier con hallazgos y referencias [N#].",
    agent=researcher,
)

analysis_task = Task(
    description=f"Analiza el dossier en hasta 150 palabras. Separa hechos e interpretaciones. KB: {KB_TEXT}\n{RULES}",
    expected_output="Análisis con referencias y dudas.",
    agent=analyst,
    context=[research_task],
)

editing_task = Task(
    description=f"Redacta un boletín de hasta 200 palabras sobre {TOPIC}. Usa dossier y análisis. KB: {KB_TEXT}\n{RULES}",
    expected_output="Boletín en Markdown con citas [N#].",
    agent=editor,
    context=[research_task, analysis_task],
)

crew = Crew(
    agents=[researcher, analyst, editor],
    tasks=[research_task, analysis_task, editing_task],
    process=Process.sequential,
    verbose=True,
)

### Ejecutar
Esta celda ejecuta los tres agentes en orden. Puede tardar varios minutos.

In [10]:
# Ejecuta la Crew
result = await crew.kickoff_async(inputs={"topic": TOPIC})
bulletin = result.raw
print("Boletín generado.")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c00f83fd-db2e-4b75-b29b-11a8f23f330a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Tema: Diplomacia, conflictos y cooperación internacional. Llama a consultar_kb para leer las noticias.   │
│                                                                                                                 │
│  Escribe en español. Usa solo las noticias proporcionadas.                                                      │
│  No inventes datos. Cita los identificadores [N1], [N2], etc.                                                   │
│  Distingue entre hechos reportados e interpretaciones.                                                          │
│  Solo has leído resúmenes RSS, no artículos completos.                                                          │
│                                                                                                                 │
│  Prepara un dossier de hasta 180 palabras con hechos y citas [N#].                                              │
│  ID: 80a2cba3-47a6-44f3-9c36-a42b3bd0afd3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador                                                                                            │
│                                                                                                                 │
│  Task: Tema: Diplomacia, conflictos y cooperación internacional. Llama a consultar_kb para leer las noticias.   │
│                                                                                                                 │
│  Escribe en español. Usa solo las noticias proporcionadas.                                                      │
│  No inventes datos. Cita los identificadores [N1], [N2], etc.                                                   │
│  Distingue entre hechos reportados e interpretaciones.                                                          │
│  Solo has leído resúmenes RSS, no artículos completos.                                                          │
│                                                                                                                 │
│  Prepara un dossier de hasta 180 palabras con hechos y citas [N#].                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool consultar_kb executed with result: [N1] El Niño pone a prueba la capacidad del mundo para adaptarse al cambio climático
URL: https://news.un.org/feed/view/es/story/2026/09/1541945
El Niño se ha fortalecido en un planeta que ya es más c...


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: consultar_kb                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Boletín de Diplomacia, Conflictos y Cooperación Internacional**                                              │
│                                                                                                                 │
│  **El Niño y el Cambio Climático**                                                                              │
│                                                                                                                 │
│  El Niño se ha fortalecido en un planeta que ya es más cálido, aumentando el riesgo de sequías, inundaciones y  │
│  pérdidas de cosechas en algunas de las principales regiones productoras de alimentos del mundo. [N1]           │
│                                                                                                                 │
│  **Reconfiguración del Poder y Multilateralismo**                                                               │
│                                                                                                                 │
│  El titular de la ONU, António Guterres, alertó de un mundo en plena reconfiguración del poder y reclamó un     │
│  multilateralismo más representativo capaz de someter el poder a las reglas y al derecho internacional. [N2]    │
│                                                                                                                 │
│  **Asamblea General de la ONU**                                                                                 │
│                                                                                                                 │
│  La Asamblea General de la ONU (UNGA81) se reunió para debatir temas prioritarios como la cuestión de           │
│  Palestina, la guerra de Ucrania, el cambio climático y la justicia social. [N3]                                │
│                                                                                                                 │
│  **Análisis y Dudas**                                                                                           │
│                                                                                                                 │
│  * El Niño se ha fortalecido en un planeta que ya es más cálido, aumentando el riesgo de sequías, inundaciones  │
│  y pérdidas de cosechas en algunas de las principales regiones productoras de alimentos del mundo. [N1]         │
│  * El titular de la ONU, António Guterres, alertó de un mundo en plena reconfiguración del poder y reclamó un   │
│  multilateralismo más representativo capaz de someter el poder a las reglas y al derecho internacional. [N2]    │
│  * La Asamblea General de la ONU (UNGA81) se reunió para debatir temas prioritarios como la cuestión de         │
│  Palestina, la guerra de Ucrania, el cambio climático y la justicia social. [N3]                                │
│                                                                                                                 │
│  **Referencias**                                                                                                │
│                                                                                                                 │
│  [N1] El Niño pone a prueba la capacidad del mundo para adaptarse al cambio climático                           │
│  [N2] Guterres alerta de un mundo en plena reconfigurac

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Redacta un boletín de hasta 200 palabras sobre Diplomacia, conflictos y cooperación internacional. Usa   │
│  dossier y análisis. KB: [N1] El Niño pone a prueba la capacidad del mundo para adaptarse al cambio climático   │
│  URL: https://news.un.org/feed/view/es/story/2026/09/1541945                                                    │
│  El Niño se ha fortalecido en un planeta que ya es más cálido, aumentando el riesgo de sequías, inundaciones y  │
│  pérdidas de cosechas en algunas de las principales regiones productoras de alimentos del mundo. Pero, a        │
│  diferencia de muchos desastres, este puede verse venir. Un experto de la ONU asegura que los próximos meses    │
│  ofrecen a los países una oportunidad no solo para proteger nuestra seguridad alimentaria, sino también para    │
│  prepararse para el futuro de un clima cada vez más impredecible.                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  [N2] Guterres alerta de un mundo en plena reconfiguración del poder: entre Estados, corporaciones y máquinas   │
│  URL: https://news.un.org/feed/view/es/story/2026/09/1541943                                                    │
│  Ante los líderes mundiales, el titular de la ONU, que este año concluye su mandato, dijo que mientras se       │
│  produce ese cambio de poder, las instituciones internacionales siguen ancladas en otra época. Señaló cuatro    │
│  desafíos para el futuro: la guerra, la desigualdad en todas sus formas, la crisis climática y la inteligencia  │
│  artificial. Para superarlos reclamó un multilateralismo más representativo capaz de someter el poder a las     │
│  reglas y al derecho internacional.                                                                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  [N3] Minuto a minuto de la Asamblea General UNGA81                                                             │
│  URL: https://news.un.org/feed/view/es/story/2026/09/1541929                                                    │
│  Como cada septiembre, los líderes del mundo se disponen a hacer un respaso de una agenda internacional         │
│  ocupada con temas priorarios desde hace años, como la cuestión de Palestina, la guerra de Ucrania, el cambio   │
│  climático o la justicia social, y a la que se añaden nuevos asuntos como la guerra de Irán o la inteligencia   │
│  artificial. Más allá del debate general, este año se prestará especial atención al aumento del nivel del mar   │
│  o al arma nuclear.                                                                                             │
│                                                                                                                 │
│  Escribe en español. Usa solo las noticias proporcionadas.                                                      │
│  No inventes datos. Cita los identificadores [N1], [N2], etc.                                                   │
│  Distingue entre hechos reportados e interpretaciones. 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c00f83fd-db2e-4b75-b29b-11a8f23f330a                                                                       │
│  Final Output: **Boletín de Diplomacia, Conflictos y Cooperación Internacional**                                │
│                                                                                                                 │
│  **El Niño y el Cambio Climático**                                                                              │
│                                                                                                                 │
│  El Niño se ha fortalecido en un planeta que ya es más cálido, aumentando el riesgo de sequías, inundaciones y  │
│  pérdidas de cosechas en algunas de las principales regiones productoras de alimentos del mundo. [N1]           │
│                                                                                                                 │
│  **Reconfiguración del Poder y Multilateralismo**                                                               │
│                                                                                                                 │
│  El titular de la ONU, António Guterres, alertó de un mundo en plena reconfiguración del poder y reclamó un     │
│  multilateralismo más representativo capaz de someter el poder a las reglas y al derecho internacional. [N2]    │
│                                                                                                                 │
│  **Asamblea General de la ONU**                                                                                 │
│                                                                                                                 │
│  La Asamblea General de la ONU (UNGA81) se reunió para debatir temas prioritarios como la cuestión de           │
│  Palestina, la guerra de Ucrania, el cambio climático y la justicia social. [N3]                                │
│                                                                                                                 │
│  **Análisis y Dudas**                                                                                           │
│                                                                                                                 │
│  * El Niño se ha fortalecido en un planeta que ya es más cálido, aumentando el riesgo de sequías, inundaciones  │
│  y pérdidas de cosechas en algunas de las principales regiones productoras de alimentos del mundo. [N1]         │
│  * El titular de la ONU, António Guterres, alertó de un mundo en plena reconfiguración del poder y reclamó un   │
│  multilateralismo más representativo capaz de someter el poder a las reglas y al derecho internacional. [N2]    │
│  * La Asamblea General de la ONU (UNGA81) se reunió para debatir temas prioritarios como la cuestión de         │
│  Palestina, la guerra de Ucrania, el cambio climático y la justicia social. [N3]                                │
│                                                                                                                 │
│  **Referencias**                                                                                                │
│                                                                                                                 │
│  [N1] El Niño pone a prueba la capacidad del mundo para adaptarse al cambio climático                           │
│  [N2] Guterres alerta de un mundo en plena reconfigura

Boletín generado.


## 9) Mostrar resultado
Muestra el boletín con las referencias a las fuentes citadas.

In [11]:
# Muestra el boletín con referencias
import re
cited_ids = set(re.findall(r"\[(N\d+)\]", bulletin))
references = "\n".join(f"- [{doc['id']}] {doc['title']}: {doc['url']}" for doc in KB if doc["id"] in cited_ids)
final_text = bulletin + "\n\n## Fuentes\n" + references + f"\n\n*KB descargada: {snapshot['downloaded_at_utc']}.*"
display(Markdown(final_text))

**Boletín de Diplomacia, Conflictos y Cooperación Internacional**

**El Niño y el Cambio Climático**

El Niño se ha fortalecido en un planeta que ya es más cálido, aumentando el riesgo de sequías, inundaciones y pérdidas de cosechas en algunas de las principales regiones productoras de alimentos del mundo. [N1]

**Reconfiguración del Poder y Multilateralismo**

El titular de la ONU, António Guterres, alertó de un mundo en plena reconfiguración del poder y reclamó un multilateralismo más representativo capaz de someter el poder a las reglas y al derecho internacional. [N2]

**Asamblea General de la ONU**

La Asamblea General de la ONU (UNGA81) se reunió para debatir temas prioritarios como la cuestión de Palestina, la guerra de Ucrania, el cambio climático y la justicia social. [N3]

**Análisis y Dudas**

* El Niño se ha fortalecido en un planeta que ya es más cálido, aumentando el riesgo de sequías, inundaciones y pérdidas de cosechas en algunas de las principales regiones productoras de alimentos del mundo. [N1]
* El titular de la ONU, António Guterres, alertó de un mundo en plena reconfiguración del poder y reclamó un multilateralismo más representativo capaz de someter el poder a las reglas y al derecho internacional. [N2]
* La Asamblea General de la ONU (UNGA81) se reunió para debatir temas prioritarios como la cuestión de Palestina, la guerra de Ucrania, el cambio climático y la justicia social. [N3]

**Referencias**

[N1] El Niño pone a prueba la capacidad del mundo para adaptarse al cambio climático
[N2] Guterres alerta de un mundo en plena reconfiguración del poder: entre Estados, corporaciones y máquinas
[N3] Minuto a minuto de la Asamblea General UNGA81

## Fuentes
- [N1] El Niño pone a prueba la capacidad del mundo para adaptarse al cambio climático: https://news.un.org/feed/view/es/story/2026/09/1541945
- [N2] Guterres alerta de un mundo en plena reconfiguración del poder: entre Estados, corporaciones y máquinas: https://news.un.org/feed/view/es/story/2026/09/1541943
- [N3] Minuto a minuto de la Asamblea General UNGA81: https://news.un.org/feed/view/es/story/2026/09/1541929

*KB descargada: 2026-09-23T07:36:22.547231+00:00.*

### Salida de cada agente
Compara qué aportó cada uno.

In [12]:
# Muestra la salida de cada agente
for name, task in [("Investigador", research_task), ("Analista", analysis_task), ("Editor", editing_task)]:
    print(f"\n--- {name.upper()} ---\n")
    print(task.output.raw if task.output else "No ejecutado")


--- INVESTIGADOR ---

**Dossier: Diplomacia, conflictos y cooperación internacional**

**Hallazgos y referencias**

* El Niño se ha fortalecido en un planeta que ya es más cálido, aumentando el riesgo de sequías, inundaciones y pérdidas de cosechas en algunas de las principales regiones productoras de alimentos del mundo. [N1]
* El titular de la ONU, António Guterres, alertó de un mundo en plena reconfiguración del poder y reclamó un multilateralismo más representativo capaz de someter el poder a las reglas y al derecho internacional. [N2]
* La Asamblea General de la ONU (UNGA81) se reunió para debatir temas prioritarios como la cuestión de Palestina, la guerra de Ucrania, el cambio climático y la justicia social. [N3]

**Referencias**

[N1] El Niño pone a prueba la capacidad del mundo para adaptarse al cambio climático
[N2] Guterres alerta de un mundo en plena reconfiguración del poder: entre Estados, corporaciones y máquinas
[N3] Minuto a minuto de la Asamblea General UNGA81

--- AN

## 10) Guardar resultado (opcional)
Guarda el boletín en un archivo Markdown.

In [13]:
# Guarda el resultado
run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
report_file = DATA_DIR / f"boletin_{run_id}.md"
report_file.write_text(final_text, encoding="utf-8")
print("Guardado:", report_file.resolve())

Guardado: /Users/carlos/work/ponencias/talleres/Unex26/geobrief_datos/boletin_20260923_092506.md


## 11) Ejercicios para los alumnos
1. **Cambiar el tema:** modifica `TOPIC` y `KEYWORDS`. Ejecuta desde el paso 3 y compara los resúmenes seleccionados de la misma descarga.
2. **Cambiar el formato:** pide al editor un briefing de cinco puntos sin cambiar los otros agentes. Ejecuta desde el paso 8.
3. **Observar la herramienta:** compara `USE_TOOL=True` y `False` con la misma caché. ¿El investigador leyó realmente las noticias en ambos casos?
4. **Retirar una fuente:** elimina una entrada de `KB`, ejecuta desde el paso 5 y observa qué afirmaciones desaparecen.
5. **Información ausente:** pide al analista un dato que no aparece en los textos. ¿Reconoce el límite o lo inventa?

**Criterios de revisión:** se ejecutan las tres tareas en orden; el dossier contiene referencias; el análisis distingue evidencia e interpretación; el boletín no cita fuentes inexistentes; al menos tres afirmaciones se comprueban manualmente contra sus fragmentos.

**Idea clave:** CrewAI coordina el trabajo, Ollama ejecuta el modelo y la KB aporta información reciente. Los agentes no convierten automáticamente noticias en hechos verificados.


## 12) Problemas habituales

| Problema | Qué comprobar |
|---|---|
| No se instala CrewAI | Usa un entorno limpio con Python 3.11 y el kernel correcto. |
| `running event loop` al ejecutar la Crew | Recrea agentes y tareas (pasos 7 y 8) y usa `await crew.kickoff_async(...)` en la celda de ejecución. |
| No conecta con Ollama | Abre la aplicación o ejecuta `ollama serve`. |
| Error `unexpected keyword argument num_ctx` | Ejecuta la celda 6 actualizada, sin `num_ctx`, y recrea después los agentes. |
| Falta el modelo | Ejecuta `ollama pull llama3.1:latest`. |
| Solicita una clave OpenAI | Todos los agentes deben usar `llm=llm`, con `memory=False` y `planning=False`. |
| El feed responde con error | Se registra el fallo y no se reintenta; usa una copia previamente preparada. |
| Ya se intentó la descarga | El registro evita otra petición. Revisa su contenido y el paso 4 antes de reintentar deliberadamente. |
| No hay coincidencias | Cambia `KEYWORDS` o usa `[]`; se filtra el feed local sin nuevas peticiones. |
| Noticias antiguas | Mira la fecha original; para una nueva descarga, sigue el procedimiento del paso 4. |
| Falla el uso de herramientas | Prueba `USE_TOOL=False` y ejecuta desde el paso 3 con el feed local. |
| El informe añade detalles que no aparecen en la KB | Revisa los resúmenes y acorta las instrucciones; una referencia existente no garantiza respaldo. |

## Documentación y preparación del docente

- [Noticias ONU: feed RSS en español](https://news.un.org/feed/subscribe/es/news/all/rss.xml)
- [GDELT: aviso sobre capacidad y migración de infraestructura](https://blog.gdeltproject.org/using-the-new-web-ngrams-dataset/to-find-relevant-coverage/)
- [CrewAI: tareas y contexto](https://docs.crewai.com/en/concepts/tasks)
- [CrewAI: herramientas](https://docs.crewai.com/en/concepts/tools)
- [CrewAI: Ollama](https://docs.crewai.com/en/learn/llm-connections)
- [Ollama: Llama 3.1](https://ollama.com/library/llama3.1)

**Por qué cambia la fuente:** GDELT publicó el 30 de junio de 2026 un aviso de problemas de capacidad de su buscador durante su migración a Spanner. Propone archivos de n-gramas para búsquedas locales; procesarlos añade pasos y no proporciona el texto completo. Para mantener sencilla esta práctica se utiliza RSS.

**Preparación:** descarga el modelo, ejecuta el notebook en el equipo del curso y guarda el feed. Distribuye únicamente contenido que puedas compartir, manteniendo atribución y fecha. Revisa al menos tres afirmaciones contra los resúmenes originales.

**Validación de esta entrega:** el endpoint RSS respondió HTTP 200 con 30 entradas; se comprobó la creación de la KB con esa respuesta, la sintaxis y la reutilización de caché sin más peticiones. No se ha ejecutado el flujo de agentes contra un servidor Ollama en este entorno.
